# 11 · MCP Tools

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama. Uses this section's own MCP
> server, `03_CREWAI/mcp_server.py`, which serves the Nimbus Robotics knowledge base
> in `data/kb/`.

The **Model Context Protocol (MCP)** is the open standard for serving tools from
separate programs, so a document store, database or company API is written once
and used by any agent framework. CrewAI offers two ways to use MCP servers. This
notebook teaches the one that worked reliably in our tests, and demonstrates a
pitfall in the other.

**You will learn**
- What MCP is: servers, clients, transports
- `MCPServerAdapter`: turning an MCP server's tools into CrewAI tools, with **filtering**
- A crew answering questions from the knowledge base through MCP
- **The `Agent(mcps=...)` pitfall** in this version: invisible, unfiltered tools, and a report written on its own
- Remote servers, security, and verifying side effects

**Prerequisites:** notebook 04 (tools).

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import sys
from crewai import Agent, Task, Crew
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters
from common import SECTION_DIR, reject_leaked_tool_calls

llm = get_llm()
SERVER = StdioServerParameters(command=sys.executable, args=[str(SECTION_DIR / 'mcp_server.py')])

## 1 · Why MCP

Without a standard, every framework wraps every tool its own way. MCP separates
**who provides a tool** from **who uses it**:

```
  ┌───────────── your crew (MCP CLIENT) ─────────────┐
  │ MCPServerAdapter ──► CrewAI tools ──► Agent(tools=…) │
  └──────────┬────────────────────────────┬──────────┘
             │ stdio (a local subprocess) │ streamable HTTP / SSE (a network service)
     ┌───────▼────────┐            ┌──────▼─────────┐
     │ nimbus-kb      │            │ a remote MCP   │
     │ MCP SERVER     │            │ server         │
     └────────────────┘            └────────────────┘
```

Our server exposes four tools: `list_documents`, `read_document`,
`search_documents` and `save_report`. The six documents are about a **fictional**
company, so the model can't answer from memory; it has to use the tools.

## 2 · `MCPServerAdapter`: MCP tools as CrewAI tools

`MCPServerAdapter` (from `crewai-tools`) starts the server, connects, and gives you
its tools as ordinary CrewAI tools. It's a **context manager**: the connection and
the subprocess live for the duration of the `with` block. (A tool's `description`
also contains CrewAI's generated name and argument header; we print just the
server's description part.)

In [ ]:
with MCPServerAdapter(SERVER) as tools:
    for t in tools:
        print(f'{t.name:<18} {t.description.split("Tool Description:")[-1].strip()}')

**Filtering.** Pass tool names after the server parameters to expose only those.
Least privilege: a research crew gets read tools and never sees `save_report`:

In [ ]:
QUESTION = 'What condition did Utrecht attach to its sidewalk permit? Use the knowledge-base tools.'

with MCPServerAdapter(SERVER, 'search_documents', 'read_document') as kb_tools:
    print('visible:', [t.name for t in kb_tools])
    analyst = Agent(role='Knowledge-base analyst', goal='Answer questions from the Nimbus documents',
                    backstory='You search, then read the relevant document, then answer. You never guess.',
                    llm=llm, tools=list(kb_tools))
    task = Task(description=QUESTION, expected_output='The condition, citing the document name', agent=analyst)
    out = await Crew(agents=[analyst], tasks=[task]).kickoff_async()
print('\n' + out.raw)

The crew called the server's tools and answered with the real condition: no
operation on the Oudegracht wharf paths between 11:00 and 15:00.

## 3 · A research crew over MCP

Tools from one adapter can be shared by several agents. A researcher gathers the
facts, and a writer (with no tools at all) turns them into a customer note:

In [ ]:
with MCPServerAdapter(SERVER, 'list_documents', 'search_documents', 'read_document') as kb_tools:
    researcher = Agent(role='Researcher', goal='Collect facts from the Nimbus knowledge base',
                       backstory='Thorough. Reads documents before summarising. Cites document names.',
                       llm=llm, tools=list(kb_tools))
    writer = Agent(role='Customer writer', goal='Write clear notes for customers',
                   backstory='Uses only the facts given. Never invents dates.', llm=llm)
    research = Task(description='Find why the Albers Markt rollout will miss its 30 September target.',
                    expected_output='3-4 bullet points with facts and document names', agent=researcher,
                    guardrail=reject_leaked_tool_calls, guardrail_max_retries=2)
    note = Task(description='Write a 4-sentence note to Albers Markt explaining the delay.',
                expected_output='4 sentences', agent=writer)
    out = await Crew(agents=[researcher, writer], tasks=[research, note]).kickoff_async()

for t in out.tasks_output:
    print(f'── {t.agent} ──\n{t.raw}\n')

**Why the research task has a guardrail.** The first time we ran this cell, the
researcher's "answer" was a tool call **written as text**:

```
I need to search for information about the Albers Markt rollout…
<function=search_documents> <parameter=query> Albers Markt rollout 30 September target …
```

The local model wrote the call instead of making it. CrewAI treated that text as
the task's answer, and the writer, whose context was that text, reproduced it
as the "customer note". It's the same local-model quirk the OpenAI Agents SDK
section measured (2 of 8 runs). The fix is a task guardrail from notebook 07:
`common.reject_leaked_tool_calls` rejects an answer containing tool-call markup,
and the agent retries properly. Guard the tasks whose output other tasks build on.

## 4 · A pitfall: `Agent(mcps=[...])` in this version

CrewAI also lets you put MCP servers directly on an agent: `Agent(mcps=[MCPServerStdio(...)])`.
It looks simpler. Let's run the same permit question through it, and this time
watch everything: which tools the agent *appears* to have, which tools it
*actually* calls (with an event listener from notebook 10), and whether any
files appear in the reports folder.

In [ ]:
import asyncio
from crewai.mcp import MCPServerStdio
from crewai.events import BaseEventListener
from crewai.events.types.tool_usage_events import ToolUsageStartedEvent

tool_calls = []
class ToolWatcher(BaseEventListener):
    def setup_listeners(self, bus):
        @bus.on(ToolUsageStartedEvent)
        def on_tool(source, event):
            tool_calls.append((event.tool_name, str(event.tool_args)[:60]))
watcher = ToolWatcher()

REPORTS = SECTION_DIR / 'data' / 'reports'
before = set(REPORTS.glob('*.md')) if REPORTS.exists() else set()

direct = Agent(role='Knowledge-base analyst', goal='Answer questions from the Nimbus documents',
               backstory='You search, then read the relevant document, then answer. You never guess.',
               llm=get_llm(), mcps=[MCPServerStdio(command=sys.executable, args=[str(SECTION_DIR / 'mcp_server.py')])])
task = Task(description=QUESTION, expected_output='The condition, citing the document name', agent=direct)
out = await Crew(agents=[direct], tasks=[task]).kickoff_async()
await asyncio.sleep(0.5)

print('agent.tools (what inspection shows):', [t.name for t in (direct.tools or [])])
print('tools actually called:')
for name, args in tool_calls:
    print(f'   {name}  {args}')
print('new files in data/reports:', [f.name for f in sorted(set(REPORTS.glob('*.md')) - before)] if REPORTS.exists() else [])
print('\nanswer:', out.raw[:250])

Look carefully; there are four separate problems here (observed with CrewAI 1.15):

1. **Inspection lies.** `agent.tools` is empty, yet the agent called several tools.
   The `mcps=` tools are attached to the agent's *execution*, not the object you
   inspect. Only the events show what really happened.
2. **The tool names are opaque hashes** built from the file path
   (`home_…_orchestration_0_00279f11`). The model has only the descriptions to go
   on, and your logs become unreadable.
3. **Every tool on the server was exposed, including `save_report`.** In our runs the
   agent **called `save_report` on its own** and wrote reports nobody asked for.
   (Check the "new files" line in your run.) We only noticed because unexpected
   files appeared in `data/reports`.
4. **The answer was still wrong**: it read the rollout plan instead of the permits
   document and reported "a permit was required" as the condition.

`MCPServerAdapter` with an explicit tool filter (§2–3) avoided all four: clean
names, only the tools you allowed, correct answers.

The lasting lessons go beyond this one API:

- **Verify tool access from events, not from the object.** What an agent *can* do
  is decided at runtime.
- **Least privilege is not automatic.** Unless you filter, a write tool on the server
  is a write tool the agent may use.
- **Watch the system of record.** Unexpected files, rows or emails are how
  unwanted side effects show up.

## 5 · Remote servers

A server running as a web service uses a dict instead of `StdioServerParameters`:

```python
with MCPServerAdapter({'url': 'https://tools.example.com/mcp', 'transport': 'streamable-http'}) as tools:
    ...
```

(`'transport': 'sse'` for older SSE servers.) Authentication headers and the
network trust boundary then matter as much as the tools themselves.

## 6 · Security, and verifying side effects

- **A stdio server is a program you run**, with your permissions. Trust it like a dependency.
- **Tool descriptions are prompts.** A malicious server can steer the model with them.
- **Tool outputs are untrusted input.** A document can contain instructions.
- **Least privilege:** filter tools per agent (as with the read-only crew above).
- **CrewAI has no built-in approval for tool calls.** For a tool that writes (like
  `save_report`), put the approval in a **Flow** step with `@human_feedback`
  (notebook 08), and call the write only after approval. That's what the project does.
- **Verify writes in the system of record** (the file, the database), not in the
  agent's reply.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Using adapter tools after the `with` block | errors: the connection is closed | run crews inside the block |
| Every agent gets every tool | confused agents, too much power | filter tool names per adapter |
| `Agent(mcps=...)` inside a crew (this version) | invisible, unfiltered tools; unrequested writes | `MCPServerAdapter` with a tool filter |
| Trusting `agent.tools` to show what an agent can do | hidden tools get used | watch tool-usage events |
| Writes without approval or verification | unapproved or phantom side effects | Flow approval + check the result |

## 8 · Check yourself

<details><summary><b>1.</b> What does <code>MCPServerAdapter</code> do, and why is it a context manager?</summary>

It starts/connects to an MCP server and exposes its tools as CrewAI tools. The connection (and, for stdio, the subprocess) must be opened and closed, and the `with` block guarantees cleanup.
</details>

<details><summary><b>2.</b> How do you give a research agent only the read tools of a server?</summary>

Pass the allowed tool names to the adapter: `MCPServerAdapter(params, 'search_documents', 'read_document')`.
</details>

<details><summary><b>3.</b> What went wrong with <code>Agent(mcps=...)</code> in a crew, and how did we notice?</summary>

The agent got every tool on the server (including `save_report`) under opaque hashed names, while `agent.tools` showed nothing; it wrote reports nobody asked for and still answered wrongly. We noticed through tool-usage events and unexpected files in `data/reports`, not by inspecting the agent.
</details>

<details><summary><b>4.</b> How do you require approval before an MCP tool writes something in CrewAI?</summary>

CrewAI has no per-tool approval, so do it in a Flow: a step with `@human_feedback` decides, and only the approved branch calls the write. Then verify the write in the system of record.
</details>

## Takeaway

`MCPServerAdapter` turns any MCP server into CrewAI tools: filter them per agent,
keep crews inside the `with` block, and use a Flow step for approvals before
writes. Verify what agents *actually* do from events and the system of record:
in this version `Agent(mcps=...)` gave a crew agent every server tool under
hidden, hashed names, and it wrote reports on its own.

Next → `12_production_concerns.ipynb`: limits, errors and testing.